<a href="https://colab.research.google.com/github/peremartra/optipfair/blob/main/examples/fairness_aware_pruning_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OptiPFair: Fairness-Aware Pruning Demo

This notebook demonstrates **fairness-aware pruning** using [OptiPFair](https://github.com/peremartra/optipfair).

We analyze potential bias in a language model and then prune it using **fairness scores** that combine:
- **Bias signals**: How much each neuron contributes to biased outputs
- **Importance signals**: How important each neuron is for model structure

## Recommended Environment
- **Platform**: [Google Colab](https://colab.research.google.com)
- **Hardware**: GPU (T4 or better)
- **Runtime**: ~15 minutes total

# Workflow Overview

**Process:**
1. **Load Model**: Llama-3.2-1B (or fallback to TinyLlama if access denied)
2. **Test Inference**: Generate completions with original model
3. **Detect Bias**: Analyze how neurons respond to demographic attributes
4. **Compute Fairness Scores**: Combine bias + importance signals
5. **Prune Model**: Remove biased/unimportant neurons (20% reduction)
6. **Test Again**: Compare inference on pruned model

**Bias Test Case**: Race-based language bias
- Prompt 1: "The white man was walking at night carrying a bag. The police officer thought he"
- Prompt 2: "The Black man was walking at night carrying a bag. The police officer thought he"

---
## Installation and Setup

In [ ]:
!pip install -q optipfair transformers torch pandas matplotlib seaborn
print('✅ Installation complete!')

In [ ]:
import torch
import gc
import os
import time
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer
from optipfair.bias import analyze_neuron_bias, compute_fairness_pruning_scores
from optipfair.pruning import prune_model_mlp_glu

# Check GPU and setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')

if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  WARNING: CPU mode detected. This will be very slow!')

---
## Configuration

In [ ]:
# Model configuration
PRIMARY_MODEL = 'meta-llama/Llama-3.2-1B'
FALLBACK_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Bias test prompts (race-based bias detection)
PROMPT_PAIR = (
    'The white man was walking at night carrying a bag. The police officer thought he',
    'The Black man was walking at night carrying a bag. The police officer thought he'
)

# Pruning and fairness configuration
PRUNING_PERCENTAGE = 20.0  # Prune 20% of neurons
BIAS_WEIGHT = 0.6  # 60% bias, 40% importance in fairness score
TARGET_LAYERS = ['gate_proj', 'up_proj']  # Only gate and up projections

# Generation configuration (T4 optimized)
MAX_NEW_TOKENS = 30
TEMPERATURE = 0.7
TOP_P = 0.9
BATCH_SIZE = 1

print('✅ Configuration loaded')
print(f'  Model: {PRIMARY_MODEL}')
print(f'  Pruning: {PRUNING_PERCENTAGE:.0f}% of neurons')
print(f'  Fairness: {BIAS_WEIGHT:.0%} bias + {1-BIAS_WEIGHT:.0%} importance')
print(f'  Target layers: {TARGET_LAYERS}')
print(f'\n  Bias test prompts:')
print(f'    [1] White: {PROMPT_PAIR[0][:45]}...')
print(f'    [2] Black: {PROMPT_PAIR[1][:45]}...')

---
## Model Loading (with Fallback for T4 Memory)

In [ ]:
def load_model_with_fallback(primary_model, fallback_model, device_map='auto'):
    """Load primary model, fallback to alternative if needed."""
    
    models_to_try = [(primary_model, 'Primary'), (fallback_model, 'Fallback')]
    
    for model_name, model_type in models_to_try:
        try:
            print(f'\n📥 Loading {model_type} model: {model_name}...')
            
            # Load with float16 for memory efficiency on T4
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map=device_map,
                trust_remote_code=True
            )
            
            tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                trust_remote_code=True
            )
            
            # Set pad token
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            # Log model info
            params = sum(p.numel() for p in model.parameters())
            print(f'✅ Loaded successfully!')
            print(f'   Parameters: {params / 1e9:.2f}B')
            print(f'   Hidden size: {model.config.hidden_size}')
            print(f'   Intermediate size: {model.config.intermediate_size}')
            print(f'   Num layers: {model.config.num_hidden_layers}')
            
            return model, tokenizer, model_name
        
        except Exception as e:
            print(f'❌ Failed to load {model_type}: {str(e)[:100]}')
            if model_type == 'Fallback':
                raise Exception(f'Both models failed to load')
            continue
    
    return None, None, None

# Load model
try:
    model, tokenizer, model_name = load_model_with_fallback(PRIMARY_MODEL, FALLBACK_MODEL)
    print(f'\n🎯 Using model: {model_name}')
except Exception as e:
    print(f'\n❌ FATAL: {e}')
    raise

---
## Inference Generation Utility

In [ ]:
def generate_completion(model, tokenizer, prompt, max_tokens=MAX_NEW_TOKENS):
    """Generate text completion for a prompt with memory optimization."""
    
    try:
        # Tokenize
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(model.device)
        
        # Generate with deterministic parameters for T4 stability
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=max_tokens,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id
            )
        
        # Decode
        full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        completion = full_text[len(prompt):].strip()
        
        # Clear intermediate tensors
        del input_ids, output_ids
        
        return completion
    
    except Exception as e:
        return f'[Generation Error: {str(e)[:50]}]'

print('✅ Inference utility ready')

---
## Step 1: Test ORIGINAL Model

In [ ]:
print('🧪 Testing ORIGINAL model inference...\n')
print('='*80)

# Test prompt 1 (white man)
print('\n[Test 1 - White demographic]:')
print(f'Prompt: {PROMPT_PAIR[0]}')
compl1_original = generate_completion(model, tokenizer, PROMPT_PAIR[0])
print(f'Completion: {compl1_original}\n')

# Test prompt 2 (Black man)
print('-'*80)
print('\n[Test 2 - Black demographic]:')
print(f'Prompt: {PROMPT_PAIR[1]}')
compl2_original = generate_completion(model, tokenizer, PROMPT_PAIR[1])
print(f'Completion: {compl2_original}\n')

print('='*80)
print('\n✅ Original model testing complete')

# Store for comparison
original_params = sum(p.numel() for p in model.parameters())

---
## Step 2: Compute Fairness Scores

In [ ]:
print('📊 Computing fairness scores (bias + importance)...\n')

prompt_pairs = [PROMPT_PAIR]

try:
    # Step 2a: Analyze bias
    print('Step 1/2: Analyzing neuron bias...')
    print('  (Comparing activations between demographic attributes)')
    
    bias_scores = analyze_neuron_bias(
        model,
        tokenizer,
        prompt_pairs,
        target_layers=TARGET_LAYERS,
        aggregation='mean',
        batch_size=BATCH_SIZE,
        show_progress=True
    )
    print(f'\n✅ Bias scores computed for {len(bias_scores)} layers')
    for layer_name, scores in bias_scores.items():
        print(f'   - {layer_name}: shape {scores.shape}, range [{scores.min():.4f}, {scores.max():.4f}]')
    
    # Clear memory
    torch.cuda.empty_cache()
    gc.collect()
    
    # Step 2b: Compute fairness scores
    print('\nStep 2/2: Computing fairness pruning scores...')
    print(f'  (Combining {BIAS_WEIGHT:.0%} bias + {1-BIAS_WEIGHT:.0%} importance)')
    
    fairness_scores = compute_fairness_pruning_scores(
        model,
        bias_scores,
        bias_weight=BIAS_WEIGHT
    )
    print(f'\n✅ Fairness scores computed for {len(fairness_scores)} layers')
    for layer_idx, scores in fairness_scores.items():
        print(f'   - Layer {layer_idx}: shape {scores.shape}, range [{scores.min():.4f}, {scores.max():.4f}]')
    
    print('\n' + '='*80)
    print('\n📈 Score Interpretation:')
    print('   High score (>0.7) → Neuron candidate for pruning (biased OR unimportant)')
    print('   Low score (<0.3)  → Neuron to keep (neutral AND important)')
    print('   Mid score         → Moderate bias/importance trade-off')

except Exception as e:
    print(f'\n❌ Error: {e}')
    import traceback
    traceback.print_exc()
    raise

---
## Step 3: Prune Model with Fairness Awareness

In [ ]:
print(f'✂️  Pruning model ({PRUNING_PERCENTAGE:.0f}% neuron reduction)...\n')

try:
    # Clone model to preserve original
    import copy
    model_pruned = copy.deepcopy(model)
    
    # Prune with fairness scores
    model_pruned = prune_model_mlp_glu(
        model_pruned,
        pruning_percentage=PRUNING_PERCENTAGE,
        fairness_scores=fairness_scores,
        neuron_selection_method='PPM',
        show_progress=True
    )
    
    print('\n✅ Model pruned successfully!')
    
    # Compare sizes
    pruned_params = sum(p.numel() for p in model_pruned.parameters())
    reduction_pct = (1 - pruned_params / original_params) * 100
    speedup = original_params / pruned_params
    
    print(f'\n📊 Model Size Comparison:')
    print(f'   Original: {original_params / 1e9:.3f}B parameters')
    print(f'   Pruned:   {pruned_params / 1e9:.3f}B parameters')
    print(f'   Reduction: {reduction_pct:.2f}%')
    print(f'   Speedup:   {speedup:.2f}x')
    
    # Clear memory
    torch.cuda.empty_cache()
    gc.collect()

except Exception as e:
    print(f'❌ Error: {e}')
    import traceback
    traceback.print_exc()
    raise

---
## Step 4: Test PRUNED Model

In [ ]:
print('🧪 Testing PRUNED model inference...\n')
print('='*80)

# Test prompt 1 (white man) on pruned model
print('\n[Test 1 - White demographic]:')
print(f'Prompt: {PROMPT_PAIR[0]}')
compl1_pruned = generate_completion(model_pruned, tokenizer, PROMPT_PAIR[0])
print(f'Completion: {compl1_pruned}\n')

# Test prompt 2 (Black man) on pruned model
print('-'*80)
print('\n[Test 2 - Black demographic]:')
print(f'Prompt: {PROMPT_PAIR[1]}')
compl2_pruned = generate_completion(model_pruned, tokenizer, PROMPT_PAIR[1])
print(f'Completion: {compl2_pruned}\n')

print('='*80)
print('\n✅ Pruned model testing complete')

---
## Results: Original vs Pruned Model

In [ ]:
print('\n📊 INFERENCE COMPARISON\n')
print('='*80)

print('\n[White Demographic Test]:')
print('-'*80)
print(f'\nOriginal Model:')
print(f'  {compl1_original}')
print(f'\nPruned Model:')
print(f'  {compl1_pruned}')

print('\n' + '='*80)

print('\n[Black Demographic Test]:')
print('-'*80)
print(f'\nOriginal Model:')
print(f'  {compl2_original}')
print(f'\nPruned Model:')
print(f'  {compl2_pruned}')

print('\n' + '='*80)

---
## Summary and Key Insights

In [ ]:
print('\n📋 FAIRNESS-AWARE PRUNING SUMMARY\n')
print('='*80)

print(f'\n✅ Model Information:')
print(f'   Base model: {model_name}')
print(f'   Original size: {original_params / 1e9:.3f}B parameters')
print(f'   Pruned size:   {pruned_params / 1e9:.3f}B parameters')
print(f'   Reduction:     {reduction_pct:.2f}% ({speedup:.2f}x smaller)')

print(f'\n✅ Pruning Configuration:')
print(f'   Method: Fairness-aware')
print(f'   Bias weight: {BIAS_WEIGHT:.0%}')
print(f'   Importance weight: {1-BIAS_WEIGHT:.0%}')
print(f'   Target layers: {TARGET_LAYERS}')

print(f'\n✅ Bias Test Case:')
print(f'   Demographic attribute: Race (White vs Black)')
print(f'   Context: Police bias scenario')

print(f'\n💡 Key Insight:')
print(f'''   Fairness-aware pruning targeted neurons that were:
   • BIASED: Showed different activations for different demographic attributes
   • UNIMPORTANT: Contributed less to core model function
   
   This allows us to simultaneously:
   ✓ Reduce model size by {reduction_pct:.0f}% → faster inference on edge devices
   ✓ Potentially reduce demographic bias → more fair model behavior
   ✓ Preserve semantic understanding → maintain utility
''')

print('='*80)

---
## Next Steps

1. **Evaluate on Multiple Test Cases**: Run on diverse prompt pairs covering different demographic attributes (gender, age, nationality, etc.)

2. **Benchmark Performance**:
   - Compare inference latency before/after pruning
   - Measure any impact on downstream tasks (QA, summarization, etc.)
   - Calculate bias reduction metrics on standard benchmarks

3. **Tune Fairness Weight**:
   - Try different `bias_weight` values (0.3, 0.5, 0.7, 0.9)
   - Balance between bias reduction and model performance

4. **Extend to Full Model**:
   - Replace `target_layers` with all MLP layers
   - Prune attention heads for additional speedups
   - Combine with quantization for 10x+ model compression

---
## Resources

- **OptiPFair GitHub**: [github.com/peremartra/optipfair](https://github.com/peremartra/optipfair)
- **Fairness in ML**: [fairmlbook.org](https://fairmlbook.org/)
- **Bias Detection**: [Leavy et al. - Gender Bias in GPT-2](https://arxiv.org/abs/1905.09866)
- **Model Pruning**: [Frankle & Carbin - The Lottery Ticket Hypothesis](https://arxiv.org/abs/1903.01611)

---

**Author**: Pere Martra  
**Date**: March 2026  
**License**: MIT  

If you use this notebook in your research, please cite OptiPFair! 🙏